# PoC Pipeline — The Anatomy of a Viral Hit
**UE28 Big Data — Spotify Charts Analysis (2017–2021)**

This notebook is the **M1 deliverable**. It runs the full Bronze → Silver → Gold pipeline and demonstrates:
- Explicit schema enforcement on 3 heterogeneous sources
- Named cleaning transformations with metric logging
- At least 11 distinct Delta `.write()` calls
- 6 window function patterns (AA2)
- Performance measurement infrastructure (AA4)

The dashboard at `http://localhost:8050` reads exclusively from the Gold Delta tables written here.

## 0. Setup

In [ ]:
import sys, os
# Make the package importable whether running locally or in Docker
sys.path.insert(0, os.path.abspath('../src'))
os.environ.setdefault('DATA_ROOT', os.path.abspath('../data'))

from bigdata_music.spark_session import get_spark
from bigdata_music import config
from bigdata_music.utils.metrics import measure

spark = get_spark('bigdata-music-poc')
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)
print('DATA_ROOT:', config.DATA_ROOT)

---
## 1. Bronze — Schema-Enforced Raw Ingestion

Each source is read with an **explicit StructType** (`inferSchema=false`). Corrupt records are captured, not silently dropped. Three Delta writes happen here — one per source.

### 1.1 Source 1 — Spotify Charts (3.5 GB CSV)

In [ ]:
from bigdata_music.schemas import CHARTS_RAW_SCHEMA, CORRUPT_RECORD_FIELD

print('Schema:'); print(CHARTS_RAW_SCHEMA.simpleString())

schema_with_corrupt = CHARTS_RAW_SCHEMA.add(CORRUPT_RECORD_FIELD)

from pyspark.sql.functions import current_timestamp, year, to_date

df_charts_raw = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'false')          # EXPLICIT — never inferred
    .option('mode', 'PERMISSIVE')
    .option('columnNameOfCorruptRecord', '_corrupt_record')
    .option('encoding', 'UTF-8')
    .schema(schema_with_corrupt)
    .csv(str(config.CHARTS_RAW))
)

corrupt_count = df_charts_raw.filter(df_charts_raw['_corrupt_record'].isNotNull()).count()
total_count   = df_charts_raw.count()
print(f'Total rows: {total_count:,}')
print(f'Corrupt records: {corrupt_count:,} ({corrupt_count/total_count:.3%})')

In [ ]:
df_charts_raw.show(3, truncate=60)

In [ ]:
# WRITE #1 — Bronze charts
with measure(spark, 'bronze.charts'):
    df_bronze_charts = (
        df_charts_raw
        .withColumn('ingestion_ts', current_timestamp())
        .withColumn('year', year(to_date('date', 'yyyy-MM-dd')))
    )
    (
        df_bronze_charts.write
        .format('delta')
        .partitionBy('year')      # 5 partitions ~700 MB each
        .mode('overwrite')
        .save(config.BRONZE_CHARTS)
    )
    print(f'bronze.charts written — partitioned by year to {config.BRONZE_CHARTS}')

### 1.2 Source 2 — Spotify Track Features (4.3 GB Parquet)

In [ ]:
from bigdata_music.schemas import TRACK_FEATURES_RAW_SCHEMA
from pyspark.sql.functions import floor

df_features_raw = (
    spark.read
    .schema(TRACK_FEATURES_RAW_SCHEMA)       # validate file matches expectations
    .parquet(str(config.TRACK_FEATURES_RAW))
)
print(f'Track features rows: {df_features_raw.count():,}')
df_features_raw.printSchema()

In [ ]:
# WRITE #2 — Bronze track features
with measure(spark, 'bronze.track_features'):
    df_bronze_features = (
        df_features_raw
        .withColumn('ingestion_ts', current_timestamp())
        .withColumn('release_decade',
                    (floor(year('album_release_date') / 10) * 10).cast('int'))
    )
    (
        df_bronze_features.write
        .format('delta')
        .partitionBy('release_decade')  # ~6 era-partitions
        .mode('overwrite')
        .save(config.BRONZE_TRACK_FEATURES)
    )
    print(f'bronze.track_features written — partitioned by release_decade')

### 1.3 Source 3 — Country Enrichment (63-row CSV with deliberate quirks)

In [ ]:
from bigdata_music.schemas import COUNTRIES_RAW_SCHEMA

# The file has: UTF-8 BOM (stripped automatically), mixed quote styles, leading space in one field
df_countries_raw = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'false')
    .option('encoding', 'UTF-8')  # strips BOM
    .schema(COUNTRIES_RAW_SCHEMA)
    .csv(str(config.COUNTRIES_RAW))
)

# Show the GB row — should have leading space in continent (the deliberate quirk)
df_countries_raw.filter("region_code = 'gb'").show()

# WRITE #3 — Bronze countries
with measure(spark, 'bronze.countries'):
    (
        df_countries_raw.withColumn('ingestion_ts', current_timestamp())
        .write
        .format('delta')
        .mode('overwrite')
        .save(config.BRONZE_COUNTRIES)
    )
    print(f'bronze.countries written — {df_countries_raw.count()} rows')

---
## 2. Silver — Cleaning Substance

Each cleaning step is a **named function** with a docstring explaining *why*. Metric logging after each step quantifies what was cleaned.

### 2.1 Charts Cleaning

In [ ]:
from bigdata_music.silver.clean_charts import (
    extract_track_id, cast_date_with_fallback, cast_numeric_columns,
    filter_corrupt_rows, deduplicate_chart_entries, add_time_columns,
)
from pyspark.sql.functions import col

df_c = spark.read.format('delta').load(config.BRONZE_CHARTS)
print(f'Bronze charts: {df_c.count():,} rows')

df_c = extract_track_id(df_c)
print('After extract_track_id:', df_c.filter(col('track_id').isNotNull()).count(), 'with valid track_id')

df_c = cast_date_with_fallback(df_c)
df_c = cast_numeric_columns(df_c)
df_c = filter_corrupt_rows(df_c)
df_c = deduplicate_chart_entries(df_c)
df_c = add_time_columns(df_c)
df_c = df_c.drop('date', '_corrupt_record')

print(f'Silver charts: {df_c.count():,} rows')
df_c.show(3, truncate=50)

In [ ]:
# WRITE #4 — Silver charts_cleaned (partition by region, year — query-aligned)
with measure(spark, 'silver.charts_cleaned'):
    (
        df_c.write
        .format('delta')
        .partitionBy('region', 'year')
        .mode('overwrite')
        .save(config.SILVER_CHARTS_CLEANED)
    )
    print('silver.charts_cleaned written')

### 2.2 Track Features Cleaning

In [ ]:
from bigdata_music.silver.clean_features import (
    clamp_audio_features, assign_mood_quadrant,
    build_track_level_view, add_release_decade,
)
from pyspark.sql.functions import col

df_f = spark.read.format('delta').load(config.BRONZE_TRACK_FEATURES)

before = df_f.count()
df_f = df_f.filter(col('track_id').isNotNull())
print(f'Dropped {before - df_f.count():,} rows with null track_id')

df_f = clamp_audio_features(df_f)    # clamp [0,1] features
df_f = assign_mood_quadrant(df_f)    # Russell's circumplex model
df_f = add_release_decade(df_f)

print('Mood quadrant distribution:')
df_f.groupBy('mood_quadrant').count().orderBy('count', ascending=False).show()

# WRITE #5 — Silver tracks_cleaned (track-artist view)
# WRITE #6 — Silver tracks_cleaned (track-level view)
df_track = build_track_level_view(df_f)
print(f'Track-artist rows: {df_f.count():,}  Track-level rows: {df_track.count():,}')

In [ ]:
with measure(spark, 'silver.tracks_cleaned'):
    (
        df_f.write.format('delta').partitionBy('release_decade')
        .mode('overwrite').save(config.SILVER_TRACKS_CLEANED + '_track_artist')
    )
    (
        df_track.write.format('delta').partitionBy('release_decade')
        .mode('overwrite').save(config.SILVER_TRACKS_CLEANED)
    )
    print('silver.tracks_cleaned written (both views)')

### 2.3 Countries Cleaning

In [ ]:
from bigdata_music.silver.clean_countries import trim_string_columns
from pyspark.sql.functions import col

df_co = spark.read.format('delta').load(config.BRONZE_COUNTRIES)

print('Before trim — GB continent field:')
df_co.filter("region_code = 'gb'").select('region_code', 'country_name', 'continent').show()

df_co = trim_string_columns(df_co)
df_co = df_co.filter(col('region_code').isNotNull()).drop('ingestion_ts')

print('After trim — GB continent field:')
df_co.filter("region_code = 'gb'").select('region_code', 'country_name', 'continent').show()

# WRITE #7 — Silver countries
with measure(spark, 'silver.countries'):
    df_co.write.format('delta').mode('overwrite').save(config.SILVER_COUNTRIES)
    print(f'silver.countries written — {df_co.count()} rows')

### 2.4 The Big Join — charts × tracks × countries

In [ ]:
from pyspark.sql.functions import broadcast

charts   = spark.read.format('delta').load(config.SILVER_CHARTS_CLEANED)
tracks   = spark.read.format('delta').load(config.SILVER_TRACKS_CLEANED)
countries = spark.read.format('delta').load(config.SILVER_COUNTRIES)

# Pre-repartition both large sides on join key (AA4 Experiment 3)
charts  = charts.repartition(config.SHUFFLE_PARTITIONS, 'track_id')
tracks  = tracks.repartition(config.SHUFFLE_PARTITIONS, 'track_id')

# Sort-merge join: charts × tracks
joined = charts.join(tracks, on='track_id', how='left')

# Broadcast join: joined × countries (country_name matches charts.region)
enriched = joined.join(
    broadcast(countries),
    on=joined['region'] == countries['country_name'],
    how='left',
)

row_count = enriched.count()
print(f'silver.charts_enriched: {row_count:,} rows')
print('Columns:', enriched.columns[:15], '...')

In [ ]:
# WRITE #8 — Silver charts_enriched
with measure(spark, 'silver.charts_enriched'):
    (
        enriched.write
        .format('delta')
        .partitionBy('region', 'year', 'month')
        .mode('overwrite')
        .save(config.SILVER_CHARTS_ENRICHED)
    )
    print('silver.charts_enriched written')

---
## 3. Gold — Business Aggregates

Each Gold table powers exactly one dashboard page. This 1:1 mapping makes traceability obvious and every aggregate justify its existence.

**Window functions demonstrated (AA2):**
1. `row_number()` — deduplication (used in Silver and streak detection)
2. `date_sub(date, rn)` — gap-and-island streak key
3. `lag()` — week-over-week rank delta
4. `dense_rank()` — monthly artist leaderboard
5. `ntile(10)` — stream decile bucketing
6. `first_value()` / `last_value()` — chart lifespan

### 3.1 Regional Mood Seasonal

In [ ]:
from bigdata_music.gold.mood_seasonal import build_regional_mood_seasonal

# WRITE #9 — Gold regional_mood_seasonal
with measure(spark, 'gold.regional_mood_seasonal'):
    n = build_regional_mood_seasonal(spark)
    print(f'gold.regional_mood_seasonal: {n:,} rows')

df_mood = spark.read.format('delta').load(config.GOLD_REGIONAL_MOOD)
print('\nTop mood per region (2020, Summer):')
(
    df_mood.filter("year = 2020 and season = 'Summer'")
    .orderBy('mood_share', ascending=False)
    .select('region', 'mood_quadrant', 'mood_share', 'total_streams')
    .show(10)
)

### 3.2 Streak Analysis — Gap-and-Island Window Function (AA2 centrepiece)

In [ ]:
from bigdata_music.gold.streaks import build_streak_analysis

# WRITE #10 — Gold streak_analysis
with measure(spark, 'gold.streak_analysis'):
    n = build_streak_analysis(spark)
    print(f'gold.streak_analysis: {n:,} streak records')

df_streaks = spark.read.format('delta').load(config.GOLD_STREAK_ANALYSIS)
print('\nLongest #1 streaks globally:')
(
    df_streaks.filter("region = 'Global'")
    .orderBy('streak_days', ascending=False)
    .select('title', 'artist', 'region', 'streak_start', 'streak_end', 'streak_days')
    .show(10, truncate=40)
)

### 3.3 Sonic DNA by Rank Tier

In [ ]:
from bigdata_music.gold.sonic_dna import build_sonic_dna

# WRITE #11 — Gold sonic_dna_by_rank_tier
with measure(spark, 'gold.sonic_dna_by_rank_tier'):
    n = build_sonic_dna(spark)
    print(f'gold.sonic_dna_by_rank_tier: {n} rows (10 decile tiers)')

df_dna = spark.read.format('delta').load(config.GOLD_SONIC_DNA)
print('\nValence + Energy by stream decile (1=Hits, 10=Long tail):')
df_dna.select('stream_decile', 'track_count', 'mean_valence', 'mean_energy', 'avg_peak_rank').show()

### 3.4 Track Longevity

In [ ]:
from bigdata_music.gold.longevity import build_track_longevity

# WRITE #12 — Gold track_longevity
with measure(spark, 'gold.track_longevity'):
    n = build_track_longevity(spark)
    print(f'gold.track_longevity: {n:,} rows')

df_lon = spark.read.format('delta').load(config.GOLD_TRACK_LONGEVITY)
print('\nLongest chart runs globally:')
(
    df_lon.filter("region = 'Global'")
    .orderBy('chart_lifespan_days', ascending=False)
    .select('title', 'artist', 'chart_lifespan_days', 'peak_rank', 'valence', 'energy')
    .show(10, truncate=35)
)

### 3.5 Global Top Artists

In [ ]:
from bigdata_music.gold.top_artists import build_top_artists

# WRITE #13 — Gold global_top_artists
with measure(spark, 'gold.global_top_artists'):
    n = build_top_artists(spark)
    print(f'gold.global_top_artists: {n:,} rows')

df_art = spark.read.format('delta').load(config.GOLD_TOP_ARTISTS)
print('\nTop 10 artists in 2020:')
(
    df_art
    .filter("year(month_start) = 2020 and monthly_rank <= 10")
    .groupBy('artist')
    .agg({'monthly_streams': 'sum'})
    .orderBy('sum(monthly_streams)', ascending=False)
    .show(10)
)

---
## 4. Write Count Verification

The grader's observation from the previous PoC: *"Tu ne write jamais."* This cell counts every Delta write to prove it cannot recur.

In [ ]:
import os
from pathlib import Path

data_root = Path(config.DATA_ROOT)
delta_tables = []
for p in data_root.rglob('_delta_log'):
    delta_tables.append(str(p.parent).replace(str(data_root), ''))

print(f'Delta tables written: {len(delta_tables)}')
for t in sorted(delta_tables):
    print(f'  {t}')

---
## 5. Performance Measurement Summary (AA4)

Timings written to `reports/perf_metrics.jsonl` by the `measure()` context manager.

In [ ]:
import json
from pathlib import Path

metrics_file = Path('../reports/perf_metrics.jsonl')
if metrics_file.exists():
    records = [json.loads(l) for l in metrics_file.read_text().splitlines() if l.strip()]
    import pandas as pd
    df_perf = pd.DataFrame(records)
    print(df_perf[['label', 'aqe', 'wall_sec']].to_string(index=False))
else:
    print('No metrics yet — run the pipeline first')

---
## 6. Dashboard Verification

Confirms the dashboard reads from Gold Delta only — no SQLite, no CSV.

In [ ]:
sys.path.insert(0, os.path.abspath('..'))

from dashboard.data_loader import load_gold, available_tables

print('Available Gold tables:', available_tables())

df_streaks_pd = load_gold('streak_analysis')
print(f'\nStreak analysis via dashboard loader: {len(df_streaks_pd):,} rows')
print(df_streaks_pd.nlargest(5, 'streak_days')[['title', 'artist', 'region', 'streak_days']])

In [ ]:
spark.stop()
print('SparkSession stopped. Pipeline complete.')
print('Dashboard: http://localhost:8050')
print('Spark UI:  http://localhost:4040 (while Spark is running)')